In [ ]:
import importlib
from IPython.display import display, clear_output

import cv2
import numpy as np
from pathlib import Path
from datetime import datetime

import src.gui as gui_mod
import src.Utilities as utilities_mod

# Force reload so notebook uses latest function signatures
importlib.reload(gui_mod)
importlib.reload(utilities_mod)

from src.VideoReader import ImageSequenceReader
from src.SpatialSmoothing import SpatialSmoothing, BoxFilter, Gaussian2DFilter
from src.TemporalDerivative import (
    Temporal_Derivative,
    SimpleTemporalDifference,
    OneDCenteredDifference,
    OneDDerivativeOfGaussian,
)
from src.Threshold import Threshold, ThresholdStrategy, ManualThreshold, AdaptiveThreshold

build_gui = gui_mod.build_gui
play_side_by_side = utilities_mod.play_side_by_side
deriv_to_u8 = utilities_mod.deriv_to_u8


def _safe_name(s: str) -> str:
    keep = []
    for ch in str(s):
        if ch.isalnum() or ch in ('-', '_', '.'):
            keep.append(ch)
        else:
            keep.append('_')
    out = ''.join(keep).strip('_')
    return out or 'unknown'


def _save_frame_stack(folder: Path, frames, ext='.png'):
    folder.mkdir(parents=True, exist_ok=True)
    for i, frame in enumerate(frames):
        cv2.imwrite(str(folder / f"frame_{i:05d}{ext}"), frame)


def _save_aligned_frames(folder: Path, aligned_indices, frames, ext='.png'):
    folder.mkdir(parents=True, exist_ok=True)
    for j, (idx, frame) in enumerate(zip(aligned_indices, frames)):
        cv2.imwrite(str(folder / f"frameidx_{int(idx):05d}_seq_{j:05d}{ext}"), frame)


def run_pipeline_and_play(cfg, out):
    # ---------- 1) Read frames ----------
    reader = ImageSequenceReader(cfg["video_path"])
    orig_frames, gray_frames = reader.frames_bgr_and_gray()

    # ---------- 2) Smoothing ----------
    s_mode = cfg["smoothing"]["mode"]
    temporal_input_frames = gray_frames

    if s_mode == "box3":
        smoothing_name = "box3"
        temporal_input_frames = SpatialSmoothing(BoxFilter(kernel_size=3)).process(gray_frames)
    elif s_mode == "box5":
        smoothing_name = "box5"
        temporal_input_frames = SpatialSmoothing(BoxFilter(kernel_size=5)).process(gray_frames)
    elif s_mode == "sigma":
        s_sigma = cfg["smoothing"]["smoothing_sigma"]
        smoothing_name = f"gaussian_sigma_{s_sigma:g}"
        temporal_input_frames = SpatialSmoothing(Gaussian2DFilter(sigma=s_sigma)).process(gray_frames)
    else:
        smoothing_name = "none"

    # ---------- 3) Temporal derivative ----------
    td_mode = cfg["temporal"]["mode"]

    def make_td_strategy(mode, sigma=None, k=3):
        if mode in ("op1d", "simple"):
            return SimpleTemporalDifference()
        if mode == "centered":
            return OneDCenteredDifference()
        if mode == "dog":
            if sigma is None:
                raise ValueError("temporal_sigma is required for DoG mode")
            return OneDDerivativeOfGaussian(sigma=sigma, k=k)
        raise ValueError(f"Unknown temporal mode: {mode}")

    td_strategy = make_td_strategy(td_mode, sigma=cfg["temporal"].get("temporal_sigma"))
    td = Temporal_Derivative(td_strategy)

    idx_list = []
    deriv_list = []
    for idx, d in td.apply(enumerate(temporal_input_frames)):
        idx_list.append(int(idx))
        deriv_list.append(d)

    if not deriv_list:
        with out:
            clear_output(wait=True)
            print("No temporal derivative frames produced for this configuration.")
        return

    if td_mode == "dog":
        td_name = f"dog_sigma_{cfg['temporal'].get('temporal_sigma'):g}"
    else:
        td_name = td_mode

    # ---------- 4) Threshold ----------
    t_mode = cfg["threshold"]["mode"]

    def make_threshold_strategy(mode: str) -> ThresholdStrategy:
        if mode == "manual":
            t_value = int(cfg["threshold"]["threshold_value"])
            return ManualThreshold(threshold_value=t_value)
        if mode == "adaptive":
            return AdaptiveThreshold()
        raise ValueError(f"Unknown threshold mode: {mode}")

    thr = Threshold(make_threshold_strategy(t_mode))
    mask_by_idx = {}
    for idx, m in thr.apply(zip(idx_list, deriv_list)):
        mask_by_idx[int(idx)] = m

    if not mask_by_idx:
        with out:
            clear_output(wait=True)
            print("No threshold frames produced for this configuration.")
        return

    if t_mode == "manual":
        thr_name = f"manual_{int(cfg['threshold']['threshold_value'])}"
    else:
        thr_name = "adaptive"

    # ---------- 5) Build aligned streams for saving + playback ----------
    aligned_indices = []
    aligned_deriv = []
    aligned_mask = []
    aligned_overlay = []

    for j, i in enumerate(idx_list):
        if i not in mask_by_idx:
            continue
        mask = mask_by_idx[i].astype(np.uint8)
        masked = cv2.bitwise_and(orig_frames[i], orig_frames[i], mask=mask)

        aligned_indices.append(i)
        aligned_deriv.append(deriv_list[j])
        aligned_mask.append(mask)
        aligned_overlay.append(masked)

    # ---------- 6) Save outputs in organized folders ----------
    output_root = Path(r"F:\ML\CV\output")
    video_name = _safe_name(Path(cfg["video_path"]).name)
    run_tag = datetime.now().strftime("%Y%m%d_%H%M%S")

    run_folder_name = _safe_name(
        f"{run_tag}__{video_name}__s_{smoothing_name}__td_{td_name}__th_{thr_name}"
    )
    run_dir = output_root / run_folder_name

    orig_dir = run_dir / "original_frames"
    gray_dir = run_dir / "grayscale_frames"
    smooth_dir = run_dir / f"{_safe_name(smoothing_name)}_smoothing_frames"
    td_dir = run_dir / f"{_safe_name(td_name)}_temporal_derivatives_frames"
    thr_dir = run_dir / f"{_safe_name(thr_name)}_threshold_frames"
    overlay_dir = run_dir / "overlayed_frames"

    _save_frame_stack(orig_dir, orig_frames)
    _save_frame_stack(gray_dir, gray_frames)
    _save_frame_stack(smooth_dir, temporal_input_frames)
    _save_aligned_frames(td_dir, aligned_indices, aligned_deriv)
    _save_aligned_frames(thr_dir, aligned_indices, aligned_mask)
    _save_aligned_frames(overlay_dir, aligned_indices, aligned_overlay)

    # ---------- 7) Playback views ----------
    def six_view_generator():
        for j, i in enumerate(idx_list):
            if i not in mask_by_idx:
                continue

            orig = orig_frames[i]
            gray = gray_frames[i]
            smooth = temporal_input_frames[i]
            deriv = deriv_to_u8(deriv_list[j])
            mask = mask_by_idx[i].astype(np.uint8)
            masked = cv2.bitwise_and(orig, orig, mask=mask.astype(np.uint8))

            yield i, [orig, gray, smooth, deriv, mask, masked], [
                "Original",
                "Grayscale",
                "Smoothing",
                "1D Derivative",
                "Threshold Mask",
                "Masked Output",
            ]

    with out:
        print(f"Saved run outputs to: {run_dir}")

    play_side_by_side(six_view_generator(), fps=30, figsize=(21, 4), out=out)


VIDEO_OPTIONS = utilities_mod.list_sequence_folders(r"F:\ML\CV\Database")
ui, get_config, wait_for_run, out = build_gui(VIDEO_OPTIONS, on_run=run_pipeline_and_play)
display(ui)

